# Data Cleaning and Preparation

**Project:** E-commerce Sales Intelligence  
**Source:** [UCI Online Retail dataset](https://archive.ics.uci.edu/dataset/352/online+retail)  
**Input:** `02_Data/Raw/Online_Retail.xlsx`  
**Generated files:** `02_Data/Processed/`

This notebook documents the preparation of historical retail transactions. Each decision is based on a specific data-quality or analytical consideration, rather than automatically treating every unusual record as invalid.

The output datasets are **intermediate preparation stages**. Published business metrics use additional filters in the project's [BigQuery SQL analyses](../04_SQL_Analysis/README.md); exploratory row counts and revenue totals in Python should not be interpreted as the final dashboard KPIs.

In [1]:
from pathlib import Path

import pandas as pd

# Locate the UCI workbook across common VS Code/Jupyter working directories.
project_folder = "Project-01-Ecommerce-Sales-Intelligence"
raw_file = Path("02_Data") / "Raw" / "Online_Retail.xlsx"
working_dir = Path.cwd().resolve()
candidate_roots = [
    working_dir,
    working_dir.parent,
    working_dir / project_folder,
]
file_path = next(
    (root / raw_file for root in candidate_roots if (root / raw_file).is_file()),
    None,
)
if file_path is None:
    raise FileNotFoundError(
        "Online_Retail.xlsx not found. Download the UCI Online Retail dataset "
        "and place it in 02_Data/Raw/ in the project folder."
    )
print("Using source:", file_path)

df = pd.read_excel(file_path, sheet_name="Online Retail")

Using source: C:\Users\grin-\OneDrive\Dokumente\Analytics-Studio\Project-01-Ecommerce-Sales-Intelligence\02_Data\Raw\Online_Retail.xlsx


In [2]:
df.shape

(541909, 8)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [4]:
df.duplicated().sum()

np.int64(5268)

In [5]:
df_clean = df.drop_duplicates().copy()

In [6]:
df_clean.shape

(536641, 8)

## 1. Exact Duplicate Records

**5,268 exact duplicate rows were identified and removed.**

**Reason:** Identical transaction records could cause transactions and revenue to be counted more than once.

**Result:** The dataset was reduced from **541,909 rows to 536,641 rows**.

In [7]:
df_clean.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135037
Country             0
dtype: int64

## 2. Missing Values

After removing duplicates, **135,037 records have no `CustomerID`** and **1,454 records have no `Description`**. (The original raw data contained 135,080 missing customer IDs.)

**Decision:** Retain these records at this stage. Missing customer identifiers prevent customer-level attribution, but the associated transactions can still contribute to overall sales metrics. Missing product descriptions do not, by themselves, establish that a row is invalid.

Customer-specific analyses must count only identifiable customers and distinguish them from transactions with missing identifiers.

In [8]:
(df_clean["Quantity"] < 0).sum()

np.int64(10587)

In [9]:
(df_clean["Quantity"] > 0).sum()

np.int64(526054)

In [10]:
(df_clean["Quantity"] == 0).sum()

np.int64(0)

In [11]:
(df_clean["InvoiceNo"].astype(str).str.startswith("C") & (df_clean["Quantity"] < 0)).sum()

np.int64(9251)

In [12]:
(~df_clean["InvoiceNo"].astype(str).str.startswith("C") & (df_clean["Quantity"] < 0)).sum()

np.int64(1336)

## 3. Negative Quantities

Following duplicate removal, **10,587 rows have negative quantities**:

- **9,251** belong to invoice numbers beginning with `C`.
- **1,336** have invoice numbers that do not begin with `C`.
- **No rows** have `Quantity = 0`.

**Decision:** Retain negative quantities in the general cleaned dataset so that they remain available for inspection. Such values can reflect cancellations, returns, or operational adjustments and should not be treated as equivalent without investigation.

**Revenue-analysis rule:** Restrict the later revenue dataset to `Quantity > 0`.

In [13]:
df_clean["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(9251)

In [14]:
df_clean.loc[
    df_clean["InvoiceNo"].astype(str).str.startswith("C"),
    "InvoiceNo"
].nunique()

3836

In [15]:
df_clean.loc[
    df_clean["InvoiceNo"].astype(str).str.startswith("C"),
    "Quantity"
].describe()

count     9251.000000
mean       -29.787050
std       1147.997592
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

## 4. Cancellation Invoices

There are **9,251 rows** with an `InvoiceNo` starting with `C`, representing **3,836 distinct cancellation invoices**. Their recorded quantities are negative. Not all negative-quantity rows, however, carry the `C` prefix.

**Decision:** Preserve these rows in `df_clean`, but exclude invoices beginning with `C` when preparing the non-cancellation dataset (`df_orders`). This invoice-based exclusion does not itself reconcile all possible returns or refunds.

In [16]:
(df_clean["UnitPrice"] == 0).sum()

np.int64(2510)

In [17]:
(df_clean["UnitPrice"] <= 0).sum()

np.int64(2512)

In [18]:
df_clean.loc[
    df_clean["UnitPrice"] == 0,
    "Description"
].value_counts().head(10)

Description
check                            159
?                                 47
damages                           45
damaged                           43
found                             25
sold as set on dotcom             20
adjustment                        16
Damaged                           14
FRENCH BLUE METAL DOOR SIGN 1      9
thrown away                        9
Name: count, dtype: int64

In [19]:
(df_clean["UnitPrice"] < 0).sum()

np.int64(2)

In [20]:
df_clean[df_clean["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [21]:
# Flag descriptive operational adjustments only when their price is zero.
operational_zero_price_mask = (
    (df_clean["UnitPrice"] == 0)
    & df_clean["Description"].str.contains(
        "check|damage|damaged|damages|adjustment|thrown away|unsaleable|destroyed|lost|missing|smashed",
        case=False,
        na=False,
    )
)
zero_price_operational = df_clean.loc[operational_zero_price_mask]
zero_price_operational.shape

(368, 8)

## 5. Zero-Price and Negative-Price Records

The original raw dataset contains **2,515 zero-price rows**. After removing exact duplicates, **2,510 rows have `UnitPrice = 0`**, and **two additional records have a negative unit price** (bad-debt adjustment records with stock code `B`).

Zero-price records include a mixture of product-related lines and descriptions suggesting operational adjustments. Zero price alone is not sufficient evidence of an invalid transaction.

**Decision:** Exclude **368 zero-price operational records** identified by the documented description-based rule, while retaining the remaining **2,142 zero-price records** at this stage. One more zero-price record is excluded by the subsequent `B`/`S` stock-code rule, leaving **2,141 zero-price records** in `df_sales`. The negative-price `B` records are handled by the same stock-code rule.

The text-matching rule is a targeted analytical choice; it does not establish that every retained zero-price record represents a completed sale.

In [22]:
df_sales = df_clean.loc[~operational_zero_price_mask].copy()
df_sales.shape

(536273, 8)

In [23]:
df_sales.loc[
    df_sales["StockCode"].astype(str).str.len() == 1,
    "StockCode"
].value_counts()

StockCode
M    566
D     77
S     62
B      3
m      1
Name: count, dtype: int64

In [24]:
df_sales.loc[
    df_sales["StockCode"] == "D",
    ["InvoiceNo", "Description", "Quantity", "UnitPrice"]
].drop_duplicates()

,InvoiceNo,Description,Quantity,UnitPrice
141,C536379,Discount,-1,27.50
9038,C537164,Discount,-1,29.29
14498,C537597,Discount,-1,281.00
19392,C537857,Discount,-1,267.12
31134,C538897,Discount,-1,5.76
...,...,...,...,...
479868,C577227,Discount,-1,19.82
479869,C577227,Discount,-1,16.76
493613,C578239,Discount,-1,26.33
516221,C579884,Discount,-1,20.53


In [25]:
df_sales = df_sales.loc[
    ~df_sales["StockCode"].isin(["B", "S"])
].copy()

df_sales.shape

(536208, 8)

## 6. Sales-Specific Non-Sales Records

The sales-preparation stage excludes:

- **368** zero-price lines with descriptions matching the operational-adjustment rule above.
- **3** records with stock code `B` (bad-debt adjustments).
- **62** records with stock code `S` (samples).

The `B` and `S` exclusions remove **65 additional records**. Other codes, including `M`, `m`, and `D`, are retained at this preparation stage; `C`-prefixed invoices are handled separately below. Further exclusions of specific stock codes are defined in the final SQL business-metric queries.

**Result:** `df_sales` contains **536,208 rows** before the separate large-invoice exclusion.

In [26]:
invoice_summary = df_sales.groupby("InvoiceNo").agg(
    line_count=("InvoiceNo", "size"),
    customer_count=("CustomerID", "count")
)

invoice_summary.head()

,line_count,customer_count
InvoiceNo,,
536365,7,7
536366,2,2
536367,12,12
536368,4,4
536369,1,1


In [27]:
large_no_customer = invoice_summary[
    (invoice_summary["line_count"] >= 500) &
    (invoice_summary["customer_count"] == 0)
].index

large_no_customer

Index([536544, 536592, 536876, 537237, 537240, 537434, 537638, 537666, 537823,
       538071, 538177, 538349, 539437, 539958, 540551, 558475, 573585, 575176,
       575477, 575875, 575930, 576329, 576617, 576618, 576837, 576840, 577078,
       577358, 577768, 578344, 578347, 578827, 578844, 579187, 579508, 579512,
       579777, 580115, 580367, 580729, 580730, 580983, 581217, 581219, 581439,
       581492],
      dtype='object', name='InvoiceNo')

In [28]:
df_sales = df_sales.loc[
    ~df_sales["InvoiceNo"].isin(large_no_customer)
].copy()

df_sales.shape

(508576, 8)

In [29]:
df_sales["InvoiceNo"].isin(large_no_customer).sum()

np.int64(0)

## 7. Large Invoices Without Customer Identifiers

The invoice-level check identified **46 invoice numbers** with at least 500 transaction lines and **no non-missing `CustomerID`** values.

**Decision:** Exclude these invoice groups from the analysis-preparation dataset `df_sales`, removing **27,632 rows**. This is a **project-specific heuristic**, not proof that those invoices were erroneous or fraudulent. Large unidentified orders could reflect legitimate activity, so this exclusion is an analytical limitation worth documenting.

**Result:** `df_sales` contains **508,576 rows** after this step. No rows belonging to the 46 identified invoice numbers remain.

In [30]:
df_orders = df_sales.loc[
    ~df_sales["InvoiceNo"].astype(str).str.startswith("C")
].copy()

df_orders.shape

(499385, 8)

In [31]:
df_orders["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(0)

## 8. Non-Cancellation Invoice Dataset

`df_orders` is created by removing invoices whose `InvoiceNo` begins with `C` from the prepared `df_sales` dataset.

**Result:** `df_orders` contains **499,385 rows** with **no `C`-prefixed invoice numbers**.

**Scope note:** The name `df_orders` denotes non-cancellation-invoice records; it does **not** imply that all retained lines have positive quantities. The next step explicitly applies `Quantity > 0` for revenue calculations.

In [32]:
df_revenue = df_orders.loc[
    df_orders["Quantity"] > 0
].copy()

df_revenue.shape

(498359, 8)

In [33]:
df_revenue["Revenue"] = (
    df_revenue["Quantity"] * df_revenue["UnitPrice"]
)

df_revenue.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [34]:
df_revenue[df_revenue["Revenue"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue


In [35]:
df_revenue["Revenue"].describe()

count    498359.000000
mean         20.746940
std         278.246522
min           0.000000
25%           4.130000
50%          10.080000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64

In [36]:
df_revenue[df_revenue["Revenue"] == 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,0.0
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,0.0
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom,0.0
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,0.0
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,0.0
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,0.0


In [37]:
df_revenue.loc[
    df_revenue["Revenue"] == 0,
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice
622,536414,22139,NaN,56,0.0
1970,536545,21134,NaN,1,0.0
1971,536546,22145,NaN,1,0.0
1972,536547,37509,NaN,1,0.0
1987,536549,85226A,NaN,1,0.0
1988,536550,85044,NaN,1,0.0
2024,536552,20950,NaN,1,0.0
2025,536553,37461,NaN,3,0.0
2026,536554,84670,NaN,23,0.0
4348,536765,84952C,NaN,19,0.0


In [38]:
df_revenue.loc[
    df_revenue["Revenue"] == 0,
    "Description"
].value_counts(dropna=False).head(10)

Description
NaN                                592
found                               25
FRENCH BLUE METAL DOOR SIGN 1        9
FRENCH BLUE METAL DOOR SIGN 8        8
Found                                8
amazon                               8
FRENCH BLUE METAL DOOR SIGN No       7
FRENCH BLUE METAL DOOR SIGN 4        7
OWL DOORSTOP                         7
RECIPE BOX PANTRY YELLOW DESIGN      7
Name: count, dtype: int64

In [39]:
df_revenue["Revenue"].isna().sum()

np.int64(0)

## 9. Revenue Dataset

The `df_revenue` dataset keeps rows from `df_orders` with positive quantities and calculates:

`Revenue = Quantity × UnitPrice`

**Result:** `df_revenue` contains **498,359 rows**. The calculated `Revenue` field has no missing or negative values. Some rows have `Revenue = 0` because their unit price is zero; those rows are retained because zero price alone was not enough to classify them as invalid.

**Important:** This is an **intermediate Python dataset**, not a statement of the final project revenue. The reported business KPIs are calculated in BigQuery using additional explicit filters, including stock-code exclusions.

## 10. Export Prepared Datasets

Export the four documented preparation stages as CSV files inside `02_Data/Processed/`. These generated datasets remain local; the public repository provides the notebooks and SQL scripts rather than the large CSV exports.

In [40]:
# Save exports using the located project root, not a computer-specific path.
project_root = file_path.parent.parent.parent
processed_dir = project_root / "02_Data" / "Processed"
processed_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "df_clean.csv": df_clean,
    "df_sales.csv": df_sales,
    "df_orders.csv": df_orders,
    "df_revenue.csv": df_revenue,
}

for filename, dataset in exports.items():
    export_path = processed_dir / filename
    dataset.to_csv(export_path, index=False)
    size_mb = export_path.stat().st_size / (1024 * 1024)
    print(f"{filename}: {len(dataset):,} rows, {size_mb:.2f} MB")

df_clean.csv: 536,641 rows, 45.36 MB
df_sales.csv: 508,576 rows, 43.11 MB
df_orders.csv: 499,385 rows, 42.32 MB
df_revenue.csv: 498,359 rows, 45.70 MB


## Preparation Summary

| Dataset | Description | Rows |
|---|---|---:|
| `df_clean` | Raw transactions after exact duplicate removal | 536,641 |
| `df_sales` | Further exclusions of selected non-sales rows and unusually large unidentified invoice groups | 508,576 |
| `df_orders` | `df_sales` with `C`-prefixed invoices removed | 499,385 |
| `df_revenue` | `df_orders` restricted to positive-quantity lines, with calculated `Revenue` | 498,359 |

These counts describe the notebook's preparation stages. The published revenue, order, and customer KPIs come from the [BigQuery KPI query](../04_SQL_Analysis/01_KPI_Analysis.sql) and the defined analytical filters, not from an unfiltered sum of `df_revenue`. The source dataset and exclusions place limits on interpretations of returns, customer attribution, and net revenue.